# 03 — Sector-agnostic baseline and ranked incident list

**Outcome:** produce the first artifact an operator can react to:
`ranked_incidents.csv`.

This is a transparent research baseline, not the final model. It dispatches
transformations from neutral `measurement_kind` and directions from
`anomaly_direction`; it contains no ONT/FEC/oil-pressure branches.

The critical boundary is visible:

1. read only `SPEC-CORE`;
2. save scores and ranked incidents;
3. only then, optionally mount/read `SPEC-EVAL` for offline evaluation.

## 1. Setup and choose a completed core run

Default is the telecom output from Notebook 01. Set `MODEL_SECTOR` to
`petrobras_3w` and point `MODEL_CORE_RUN_ROOT` to the Notebook 02 run to
exercise the same model path in the second sector.

In [ ]:
import json
import math
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "week1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from week1_core import CORE_VERSION, sha256_file, write_json

SECTOR = os.getenv("MODEL_SECTOR", "telecom")
default_run = (
    DRIVE_ROOT / "outputs" / "research" / f"v{CORE_VERSION}"
    / ("telecom" if SECTOR == "telecom" else "petrobras_3w")
    / ("telecom_full_v1" if SECTOR == "telecom" else "contract_challenge_v1")
)
CORE_RUN_ROOT = Path(os.getenv("MODEL_CORE_RUN_ROOT", str(default_run)))
CORE = CORE_RUN_ROOT / "SPEC-CORE"
EVALUATION = CORE_RUN_ROOT / "SPEC-EVAL"
MODEL_RUN_ID = os.getenv("MODEL_RUN_ID", f"{SECTOR}_robust_baseline_v1")
MODEL_OUTPUT = (
    DRIVE_ROOT / "outputs" / "research" / f"v{CORE_VERSION}"
    / "models" / SECTOR / MODEL_RUN_ID
)

ENTITY_LIMIT = int(os.getenv("MODEL_ENTITY_LIMIT", "20"))
HISTORY = int(os.getenv("MODEL_HISTORY", "96"))
MIN_HISTORY = int(os.getenv("MODEL_MIN_HISTORY", "24"))
SCORE_THRESHOLD = float(os.getenv("MODEL_SCORE_THRESHOLD", "4.0"))
SCORE_CAP = float(os.getenv("MODEL_SCORE_CAP", "100.0"))

if MODEL_OUTPUT.exists():
    raise FileExistsError(
        f"Use a new MODEL_RUN_ID; refusing to overwrite {MODEL_OUTPUT}"
    )
MODEL_OUTPUT.mkdir(parents=True)
display(pd.Series({
    "sector": SECTOR,
    "spec_core": str(CORE),
    "spec_eval": str(EVALUATION),
    "model_output": str(MODEL_OUTPUT),
    "entity_limit": ENTITY_LIMIT or "all",
    "history": HISTORY,
    "minimum_history": MIN_HISTORY,
    "score_threshold": SCORE_THRESHOLD,
    "score_cap": SCORE_CAP,
}, name="value").to_frame())

## 2. Load only `SPEC-CORE`

The default selects 20 leaf entities for fast iteration. Set
`MODEL_ENTITY_LIMIT=0` for all entities once the logic is stable. This cell
has no evaluation path and loads no labels.

In [ ]:
core_manifest = json.loads((CORE / "manifest.json").read_text())
catalogue = pd.read_parquet(CORE / "metric_catalogue.parquet")
registry = pd.read_parquet(CORE / "entity_registry.parquet")
relations = pd.read_parquet(CORE / "entity_relations.parquet")
gaps = pd.read_parquet(CORE / "collection_gaps.parquet")

leaf_type = "ont" if SECTOR == "telecom" else "oil_well"
leaf_entities = sorted(
    registry.loc[registry["entity_type"].eq(leaf_type), "entity_id"]
    .astype(str).unique()
)
selected_entities = (
    leaf_entities if ENTITY_LIMIT == 0
    else leaf_entities[:ENTITY_LIMIT]
)
if not selected_entities:
    raise ValueError(f"no {leaf_type!r} entities in the core run")

telemetry_parts = sorted((CORE / "telemetry").glob("part-*.parquet"))
frames = []
for part in telemetry_parts:
    frame = pd.read_parquet(part)
    frame = frame.loc[
        frame["entity_id"].astype(str).isin(selected_entities)
    ]
    if len(frame):
        frames.append(frame)
telemetry = pd.concat(frames, ignore_index=True)
telemetry["event_ts"] = pd.to_datetime(telemetry["event_ts"], utc=True)
telemetry["value"] = pd.to_numeric(telemetry["value"], errors="coerce")
telemetry["exposure"] = pd.to_numeric(
    telemetry["exposure"], errors="coerce"
)
telemetry = telemetry.merge(
    catalogue[[
        "metric_id", "measurement_kind", "anomaly_direction"
    ]],
    on="metric_id",
    how="left",
    validate="many_to_one",
)
assert telemetry["measurement_kind"].notna().all()
print(
    f"Loaded {len(telemetry):,} rows, {telemetry.entity_id.nunique()} "
    f"entities, {telemetry.metric_id.nunique()} metrics from SPEC-CORE only."
)

## 3. Neutral measurement transformation

This logic is deliberately inline because it is still a modelling choice:

- gauge, bounded fraction, and discrete state: model the observed value;
- interval count: model log rate when exposure exists, otherwise `log1p`;
- cumulative counter: model non-negative increments and suppress resets.

Exposure formulas themselves are not duplicated here; they were settled in
the translator and imported through `week1_core.py`.

In [ ]:
model_frame = telemetry.sort_values(
    ["entity_id", "metric_id", "event_ts"], kind="stable"
).copy()
model_frame["model_value"] = model_frame["value"]

interval_count = model_frame["measurement_kind"].eq("interval_count")
has_exposure = interval_count & model_frame["exposure"].gt(0)
model_frame.loc[interval_count, "model_value"] = np.log1p(
    model_frame.loc[interval_count, "value"].clip(lower=0)
)
model_frame.loc[has_exposure, "model_value"] = np.log(
    (model_frame.loc[has_exposure, "value"].clip(lower=0) + 0.5)
    / model_frame.loc[has_exposure, "exposure"]
)

cumulative = model_frame["measurement_kind"].eq("cumulative_counter")
counter_diff = (
    model_frame.loc[cumulative]
    .groupby(["entity_id", "metric_id"], sort=False)["value"]
    .diff()
)
counter_diff = counter_diff.where(counter_diff.ge(0))
model_frame.loc[cumulative, "model_value"] = np.log1p(counter_diff)

model_frame.loc[
    model_frame["quality_code"].eq("invalid"), "model_value"
] = np.nan
display(
    model_frame.groupby("measurement_kind")["model_value"]
    .agg(["count", "min", "median", "max"])
)

## 4. Robust history-only anomaly score

Each entity-metric series is compared with its own previous observations.
`shift(1)` prevents the current point from entering its baseline. The IQR
scale has a data-derived floor for constant histories. The floor is the larger
of a typical non-zero change and 1% of the series magnitude; scores are capped
at `SCORE_CAP` so a numerically flat channel cannot dominate the whole ranking.

In [ ]:
def score_one_series(group):
    group = group.sort_values("event_ts").copy()
    history = group["model_value"].shift(1).rolling(
        HISTORY, min_periods=MIN_HISTORY
    )
    group["baseline"] = history.median()
    q25 = history.quantile(0.25)
    q75 = history.quantile(0.75)
    scale = (q75 - q25) / 1.349

    differences = group["model_value"].diff().abs()
    typical_change = differences.loc[differences.gt(0)].median(skipna=True)
    typical_magnitude = group["model_value"].abs().median(skipna=True)
    relative_floor = max(
        1e-6,
        0.01 * typical_magnitude if math.isfinite(typical_magnitude) else 0,
    )
    if not math.isfinite(typical_change) or typical_change <= 0:
        typical_change = relative_floor
    fallback = max(typical_change, relative_floor)
    group["scale"] = scale.where(scale >= relative_floor, fallback)
    group["signed_score"] = (
        (group["model_value"] - group["baseline"]) / group["scale"]
    )

    direction = group["anomaly_direction"].iloc[0]
    signed = group["signed_score"]
    if direction == "increase":
        group["anomaly_score"] = signed.clip(lower=0)
    elif direction == "decrease":
        group["anomaly_score"] = (-signed).clip(lower=0)
    else:  # both or change
        group["anomaly_score"] = signed.abs()
    group["anomaly_score"] = group["anomaly_score"].clip(upper=SCORE_CAP)
    return group

scored = pd.concat(
    [score_one_series(group) for _, group in model_frame.groupby(
        ["entity_id", "metric_id"], sort=False
    )],
    ignore_index=True,
)
score_columns = [
    "event_ts", "entity_id", "metric_id", "value", "quality_code",
    "model_value", "baseline", "scale", "signed_score",
    "anomaly_score",
]
scores_to_save = scored[score_columns].copy()
scores_to_save.to_parquet(
    MODEL_OUTPUT / "anomaly_scores.parquet", index=False
)
print(
    "Saved detector output before evaluation:",
    MODEL_OUTPUT / "anomaly_scores.parquet",
)

## 5. Convert point anomalies into operator incidents

An operator does not want millions of point scores. Flagged points are
grouped by time bucket and nearest known operational domain. For telecom,
the domain is the immediate network-topology parent of the ONT. When the
source has no topology (3W), the well is its own domain.

The ranking is intentionally simple and inspectable: maximum anomaly
score, number of affected entities, number of metrics, and declared
service/customer weights.

In [ ]:
network_relations = relations.loc[
    relations.get(
        "relation_family",
        pd.Series(index=relations.index, dtype="object"),
    ).eq("network_topology")
]
child_to_parent = (
    network_relations.drop_duplicates("child_entity_id")
    .set_index("child_entity_id")["parent_entity_id"]
    .astype(str).to_dict()
    if len(network_relations) else {}
)

priorities = {}
for row in registry.itertuples(index=False):
    attributes = json.loads(row.attributes_json or "{}")
    priorities[str(row.entity_id)] = (
        float(attributes.get("service_impact_weight", 1.0) or 1.0)
        * float(attributes.get("customer_priority_weight", 1.0) or 1.0)
    )

flagged = scored.loc[scored["anomaly_score"].ge(SCORE_THRESHOLD)].copy()
flagged["domain_entity_id"] = flagged["entity_id"].map(
    lambda entity: child_to_parent.get(str(entity), str(entity))
)
cadence_seconds = float(core_manifest.get(
    "cadence_seconds",
    1 if SECTOR == "petrobras_3w" else 900,
))
bucket_frequency = "h" if cadence_seconds >= 60 else "min"
flagged["incident_bucket"] = flagged["event_ts"].dt.floor(bucket_frequency)
flagged["priority_weight"] = flagged["entity_id"].map(priorities).fillna(1.0)

incident_columns = [
    "rank", "incident_id", "incident_start", "incident_end",
    "domain_entity_id", "affected_entity_count", "affected_entities",
    "anomalous_metric_count", "anomalous_metrics", "max_score",
    "mean_score", "max_priority_weight", "rank_score",
]
incident_rows = []
for (domain, bucket), group in flagged.groupby(
    ["domain_entity_id", "incident_bucket"], sort=False
):
    affected = sorted(group["entity_id"].astype(str).unique())
    metrics = sorted(group["metric_id"].astype(str).unique())
    max_score = float(group["anomaly_score"].max())
    mean_score = float(group["anomaly_score"].mean())
    max_priority = float(group["priority_weight"].max())
    rank_score = (
        max_score
        + math.log1p(len(affected))
        + 0.25 * math.log1p(len(metrics))
        + 0.10 * math.log1p(max_priority)
    )
    incident_rows.append({
        "incident_start": group["event_ts"].min(),
        "incident_end": group["event_ts"].max(),
        "domain_entity_id": str(domain),
        "affected_entity_count": len(affected),
        "affected_entities": "|".join(affected),
        "anomalous_metric_count": len(metrics),
        "anomalous_metrics": "|".join(metrics),
        "max_score": max_score,
        "mean_score": mean_score,
        "max_priority_weight": max_priority,
        "rank_score": rank_score,
    })

if incident_rows:
    incidents = (
        pd.DataFrame(incident_rows)
        .sort_values(
            ["rank_score", "incident_start"],
            ascending=[False, True],
            kind="stable",
        )
        .reset_index(drop=True)
    )
    incidents.insert(0, "incident_id", [
        f"{SECTOR}-INC-{index:06d}"
        for index in range(1, len(incidents) + 1)
    ])
    incidents.insert(0, "rank", np.arange(1, len(incidents) + 1))
    incidents = incidents[incident_columns]
else:
    incidents = pd.DataFrame(columns=incident_columns)

incidents.to_csv(MODEL_OUTPUT / "ranked_incidents.csv", index=False)
incidents.to_parquet(MODEL_OUTPUT / "ranked_incidents.parquet", index=False)
display(incidents.head(20))
print("Saved operator list:", MODEL_OUTPUT / "ranked_incidents.csv")

## 6. Freeze modelling report — still before evaluation

At this point the detector and ranking outputs are complete. They are
reproducible without `SPEC-EVAL` mounted.

In [ ]:
modelling_report = {
    "contract_version": CORE_VERSION,
    "week1_core_sha256": sha256_file(NOTEBOOK_HOME / "week1_core.py"),
    "sector": SECTOR,
    "spec_core": str(CORE),
    "spec_eval_read_during_scoring": False,
    "selected_entity_count": len(selected_entities),
    "telemetry_rows_loaded": len(telemetry),
    "history": HISTORY,
    "minimum_history": MIN_HISTORY,
    "score_threshold": SCORE_THRESHOLD,
    "score_cap": SCORE_CAP,
    "score_rows": len(scores_to_save),
    "flagged_point_rows": len(flagged),
    "ranked_incidents": len(incidents),
    "primary_operator_artifact": "ranked_incidents.csv",
}
write_json(MODEL_OUTPUT / "modelling_report.json", modelling_report)
display(pd.Series(modelling_report, name="value").to_frame())

## 7. Optional offline evaluation — truth is read only now

This cell is not part of detection. It measures whether any flagged point
overlaps each labelled entity interval and reports simple event recall and
lead time. Condition-state coverage is reported separately because a
condition interval is not an onset/impact/resolution event.

In [ ]:
evaluation_report = {
    "status": "not_available",
    "reason": "SPEC-EVAL is absent or intentionally unmounted",
}
if EVALUATION.is_dir():
    evaluation_report = {
        "status": "evaluated_after_outputs_were_frozen",
        "spec_eval": str(EVALUATION),
    }
    interval_path = EVALUATION / "gt_fault_entity_intervals.parquet"
    if interval_path.exists():
        intervals = pd.read_parquet(interval_path)
        matched, lead_seconds = 0, []
        for row in intervals.itertuples(index=False):
            start = pd.to_datetime(row.active_start_ts, utc=True)
            end = pd.to_datetime(row.active_end_ts, utc=True)
            candidates = flagged.loc[
                flagged["entity_id"].astype(str).eq(
                    str(row.affected_entity_id)
                )
                & flagged["event_ts"].between(start, end, inclusive="left")
            ]
            if len(candidates):
                matched += 1
                impact = pd.to_datetime(
                    getattr(row, "impact_ts", pd.NaT), utc=True
                )
                if pd.notna(impact):
                    lead_seconds.append(
                        (impact - candidates["event_ts"].min()).total_seconds()
                    )
        evaluation_report.update({
            "fault_entity_intervals": len(intervals),
            "matched_fault_entity_intervals": matched,
            "fault_entity_interval_recall": (
                matched / len(intervals) if len(intervals) else None
            ),
            "median_lead_seconds": (
                float(np.median(lead_seconds)) if lead_seconds else None
            ),
        })

    condition_path = EVALUATION / "gt_condition_states.parquet"
    if condition_path.exists():
        conditions = pd.read_parquet(condition_path)
        covered = 0
        for row in conditions.itertuples(index=False):
            start = pd.to_datetime(row.condition_start_ts, utc=True)
            end = pd.to_datetime(row.condition_end_ts, utc=True)
            any_score = scored.loc[
                scored["entity_id"].astype(str).eq(str(row.entity_id))
                & scored["event_ts"].between(start, end, inclusive="left")
                & scored["anomaly_score"].notna()
            ]
            covered += int(len(any_score) > 0)
        evaluation_report.update({
            "condition_intervals": len(conditions),
            "condition_intervals_with_scored_observations": covered,
        })

write_json(MODEL_OUTPUT / "offline_evaluation.json", evaluation_report)
display(pd.Series(evaluation_report, name="value").to_frame())

## 8. What to take to an operator

Open `ranked_incidents.csv` and ask:

1. Is this the right unit of work: one device, shared domain, or service?
2. Which top-ranked rows would you investigate, suppress, or merge?
3. Which missing context would change that decision?
4. Is early warning valuable here, or only confirmed impact?

Those answers should drive the next model and contract changes. Packaging
comes later, after this output proves useful.